# Diabetes Risk Prediction
## Model Training & Evaluation
This notebook contains all model training experiments across 4 models and 4 balancing strategies (16 total combinations).

## Load the Unscaled Training Data

In [2]:
# Load preprocessed data
import numpy as np
import pandas as pd

X_train = np.load('data/X_train.npy')
X_test = np.load('data/X_test.npy')
y_train = np.load('data/y_train.npy')
y_test = np.load('data/y_test.npy')
feature_names = pd.read_csv('data/feature_names.csv').values.flatten()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Feature names:", feature_names)

X_train: (202944, 21)
y_train: (202944,)
Feature names: ['HighBP' 'HighChol' 'CholCheck' 'BMI' 'Smoker' 'Stroke'
 'HeartDiseaseorAttack' 'PhysActivity' 'Fruits' 'Veggies'
 'HvyAlcoholConsump' 'AnyHealthcare' 'NoDocbcCost' 'GenHlth' 'MentHlth'
 'PhysHlth' 'DiffWalk' 'Sex' 'Age' 'Education' 'Income']


## Training: Raw Data (No Balancing)

We start with the raw unbalanced data as our baseline.

### Random Forest on Raw Training Data

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

# Results
print("Random Forest - Raw Data Results:\n")
print(classification_report(y_test, rf_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]), 4))

Random Forest - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.96      0.91     42741
    Diabetes       0.52      0.21      0.30      7995

    accuracy                           0.84     50736
   macro avg       0.69      0.59      0.61     50736
weighted avg       0.81      0.84      0.82     50736

AUC-ROC: 0.7921


#### Random Forest - Raw Data Interpretation

The results confirm the class imbalance problem we identified during EDA. The model achieves 84% accuracy, but this is misleading because it's mostly just correctly predicting healthy patients (96% recall) while only catching 21% of actual diabetic patients.

Key takeaways:
- The model misses roughly 4 out of every 5 diabetic patients (recall = 0.21)
- When it does flag someone as diabetic, it's only right about half the time (precision = 0.52)
- The F1-score for diabetes is 0.30, which is poor
- AUC-ROC of 0.79 shows the model has some ability to distinguish between classes but struggles significantly with the minority class

This baseline result demonstrates exactly why balancing techniques like SMOTE and SMOTE-ENN are needed. Without addressing the imbalance, the model effectively learns to default to predicting "no diabetes" because that's the safe bet given the 84/16 class split.

### XGBoost on Raw Training Data
Training XGBoost with default hyperparameters on the raw unbalanced dataset.
XGBoost builds trees sequentially, where each new tree focuses on correcting the mistakes of the previous ones.

In [4]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_predictions = xgb_model.predict(X_test)

# Results
print("XGBoost - Raw Data Results:\n")
print(classification_report(y_test, xgb_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]), 4))

XGBoost - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.57      0.21      0.30      7995

    accuracy                           0.85     50736
   macro avg       0.72      0.59      0.61     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8194


#### XGBoost - Raw Data Interpretation

XGBoost performs very similarly to Random Forest on the raw data. Diabetes recall is identical at 0.21 and F1-score is the same at 0.30. XGBoost shows slight improvement in precision (0.57 vs 0.52) and AUC-ROC (0.82 vs 0.79), suggesting it ranks patients slightly better overall.

The key finding here is that switching to a more powerful model alone does not solve the class imbalance problem. Both models default to predicting "no diabetes" for the vast majority of cases. This confirms that the issue lies in the data distribution, not the model architecture, and motivates the need for resampling techniques like SMOTE and SMOTE-ENN.

## Balancing Techniques
The raw data has an 84/16 split (healthy vs diabetic). We apply three techniques to the training data only (never the test data) to address this imbalance.

The raw training data has a significant class imbalance: 170,962 healthy (class 0) vs 31,982 diabetic (class 1). We apply three balancing techniques to the training data only. The test data is never touched so it continues to represent real-world conditions.

### How each technique works:

**SMOTE (Synthetic Minority Oversampling Technique):** Creates new synthetic diabetic patients by finding pairs of real diabetic patients that are close to each other and generating a new data point somewhere in between them. This is repeated until both classes are equal in size.

**SMOTE-ENN (SMOTE + Edited Nearest Neighbors):** First applies SMOTE to create synthetic diabetic patients. Then runs ENN, which removes noisy or confusing data points from both classes. A point is removed if most of its nearest neighbors belong to a different class. This produces cleaner data but the two classes may not be perfectly equal.

**Random Undersampling:** Randomly deletes healthy patients until both classes are equal. Simple and fast but discards a large portion of the training data.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("SMOTE:", np.unique(y_train_smote, return_counts=True))

# SMOTE-ENN
smote_enn = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = smote_enn.fit_resample(X_train, y_train)
print("SMOTE-ENN:", np.unique(y_train_smoteenn, return_counts=True))

# Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print("Undersampling:", np.unique(y_train_under, return_counts=True))

print("\nOriginal:", np.unique(y_train, return_counts=True))

SMOTE: (array([0, 1]), array([170962, 170962]))
SMOTE-ENN: (array([0, 1]), array([ 98808, 159903]))
Undersampling: (array([0, 1]), array([31982, 31982]))

Original: (array([0, 1]), array([170962,  31982]))


### Balancing Results:

| Technique | Healthy (0) | Diabetic (1) | Total Samples |
|---|---|---|---|
| Original | 170,962 | 31,982 | 202,944 |
| SMOTE | 170,962 | 170,962 | 341,924 |
| SMOTE-ENN | 98,808 | 159,903 | 258,711 |
| Undersampling | 31,982 | 31,982 | 63,964 |

SMOTE doubled the dataset by generating ~139,000 synthetic diabetic patients. SMOTE-ENN also generated synthetic patients but then cleaned up noisy points from both classes, which is why the healthy count dropped from 170,962 to 98,808. The classes are not perfectly equal but the data is cleaner overall. Undersampling achieved perfect balance but at a steep cost, reducing the total dataset from 202,944 to only 63,964 samples (a 68% reduction in data).